# End-to-End ML Solution

This notebook performs EDA, preprocessing, feature engineering, model comparison, cross-validation tuning and hold-out evaluation for a tabular classification dataset.

Replace `DATA_URL` and optionally `TARGET_COLUMN` before running against the real dataset.

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'main.py').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

DATA_URL = '{{DATA_URL}}'
TARGET_COLUMN = None  # Example: 'churn'
OUTPUT_DIR = PROJECT_ROOT / 'artifacts' / 'notebook_run'
RANDOM_STATE = 42
MAX_ROWS = None

In [ ]:
import json
import warnings

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from main import (
    add_engineered_features,
    candidate_models,
    compare_models,
    evaluate_model,
    infer_target,
    load_dataset,
    make_preprocessor,
    parameter_grids,
    plot_feature_importance,
    plot_learning_curve,
    split_data,
    summarize_eda,
    tune_models,
    validate_classification_target,
)
from sklearn.preprocessing import LabelEncoder

pd.set_option('display.max_columns', 80)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

## Load Data

The next cell is guarded so the notebook can run before `DATA_URL` is replaced.

In [ ]:
if not DATA_URL or DATA_URL.startswith('{{'):
    print('Set DATA_URL to a CSV, Parquet or Excel path/URL before running the full notebook.')
    df = pd.DataFrame()
else:
    df = load_dataset(DATA_URL, max_rows=MAX_ROWS)
    print(df.shape)
    display(df.head())

## Exploratory Data Analysis

Includes summary statistics, missing values, class distribution and numeric correlations.

In [ ]:
if df.empty:
    print('EDA skipped until DATA_URL is configured.')
else:
    target_column = infer_target(df, TARGET_COLUMN)
    print(f'Target column: {target_column}')
    display(df.describe(include='all').transpose())
    display(df.isna().sum().sort_values(ascending=False).head(20).to_frame('missing_count'))
    display(df[target_column].value_counts(dropna=False).to_frame('count'))
    numeric_df = df.select_dtypes(include=[np.number])
    if not numeric_df.empty:
        display(numeric_df.corr(numeric_only=True))
    eda_summary = summarize_eda(df, target_column, OUTPUT_DIR)
    print(json.dumps(eda_summary, indent=2))

## Preprocessing And Feature Engineering

Novel features:

- `feature_missing_count`: captures row-level data quality and missingness patterns.
- `numeric_signal_mean`: captures aggregate numeric magnitude.
- `numeric_signal_std`: captures aggregate numeric variability.

In [ ]:
if df.empty:
    print('Preprocessing skipped until DATA_URL is configured.')
else:
    df_model = df.dropna(subset=[target_column]).reset_index(drop=True)
    validate_classification_target(df_model[target_column])
    label_encoder = LabelEncoder()
    y = label_encoder.fit_transform(df_model[target_column].astype(str))
    X = add_engineered_features(df_model.drop(columns=[target_column]))
    print(X[['feature_missing_count', 'numeric_signal_mean', 'numeric_signal_std']].head())
    print('Classes:', list(label_encoder.classes_))

## Train/Validation/Hold-out Split And Baseline Models

In [ ]:
if df.empty:
    print('Model comparison skipped until DATA_URL is configured.')
else:
    splits = split_data(X, y, test_size=0.2, valid_size=0.25, random_state=RANDOM_STATE)
    preprocessor = make_preprocessor(splits.X_train)
    models = candidate_models(RANDOM_STATE)
    validation_results = compare_models(preprocessor, models, splits)
    display(validation_results)

## Hyperparameter Tuning And Hold-out Evaluation

In [ ]:
if df.empty:
    print('Tuning skipped until DATA_URL is configured.')
else:
    X_train_valid = pd.concat([splits.X_train, splits.X_valid], axis=0)
    y_train_valid = np.concatenate([splits.y_train, splits.y_valid])
    best_name, best_model, tuning_results = tune_models(
        preprocessor,
        models,
        parameter_grids(),
        X_train_valid,
        y_train_valid,
        RANDOM_STATE,
    )
    holdout_metrics = evaluate_model(best_model, splits.X_holdout, splits.y_holdout)
    print('Best model:', best_name)
    display(tuning_results)
    print(json.dumps(holdout_metrics, indent=2))

## Feature Importance And Learning Curves

In [ ]:
if df.empty:
    print('Plots skipped until DATA_URL is configured.')
else:
    feature_path = OUTPUT_DIR / 'feature_importance.png'
    curve_path = OUTPUT_DIR / 'learning_curve.png'
    plot_feature_importance(best_model, splits.X_holdout, splits.y_holdout, feature_path)
    with warnings.catch_warnings():
        warnings.simplefilter('ignore')
        plot_learning_curve(best_model, X_train_valid, y_train_valid, curve_path, RANDOM_STATE)
    print(feature_path)
    print(curve_path)

## Save Model Bundle

In [ ]:
if df.empty:
    print('Model save skipped until DATA_URL is configured.')
else:
    bundle = {
        'model': best_model,
        'label_encoder': label_encoder,
        'target_column': target_column,
        'classes': list(label_encoder.classes_),
        'holdout_metrics': holdout_metrics,
    }
    model_path = OUTPUT_DIR / 'best_model.joblib'
    joblib.dump(bundle, model_path)
    print(model_path)